<a href="https://colab.research.google.com/github/rcoufms/Trabalho_IA/blob/main/coleta_arxiv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Trabalho Prático de IA - Notebook base de coleta de dados do ArXiv

**Disciplina:** Inteligência Artificial - PPGCC - FACOM/UFMS  
**Professor:** Bruno M. Nogueira  
**Semestre:** 2026/1

Este notebook é um **ponto de partida** para o trabalho prático. Ele mostra como **coletar artigos do ArXiv** filtrando por categorias e palavras-chave relacionadas ao seu tema de pesquisa, normalizar os metadados e salvar uma coleção em `JSONL`.

Você é **livre (e encorajado)** para modificar este código: trocar a API, ajustar os filtros, adicionar campos, etc. O notebook serve apenas para que você não perca tempo na parte mais mecânica do trabalho.

**Atenção:** a etapa de coleta deve ser justificada no relatório (escolha do tema, palavras-chave, categorias, janela temporal). Veja a Seção 3-(a) do enunciado e o exemplo lá fornecido.

> **A API do ArXiv é instável.** Não é incomum ver erros `HTTP 429` ("too many requests") ou `HTTP 503` ("service unavailable"), especialmente em queries longas ou em horários de pico. Este notebook implementa **retry com backoff exponencial e retomada por offset**, e o arquivo de saída preserva o progresso. Se a célula de coleta falhar, basta **rodar a mesma célula novamente** mais tarde, ela continua de onde parou. Não existe "chave de API" do ArXiv: o limite é por IP.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install rank_bm25 sentence-transformers arxiv

## 1. Instalação de dependências

Vamos usar a biblioteca [`arxiv`](https://pypi.org/project/arxiv/), um *wrapper* sobre a API oficial do ArXiv que já cuida de paginação e *rate limiting* básico. Também usamos `pandas` para inspeção tabular.

In [ ]:
# Execute apenas uma vez, descomentando se necessário.
!pip install arxiv pandas tqdm

In [ ]:
import json
import os
import time
from datetime import datetime, timezone
from pathlib import Path

import arxiv
import pandas as pd
from tqdm import tqdm

## 2. Configuração da coleta (você edita aqui!)

Adapte os campos abaixo ao **seu** tema de pesquisa. O exemplo segue o caso do enunciado ("Detecção de fraudes em transações financeiras com aprendizado profundo").

**Dica para reduzir erros 429/503:** use `PAGE_SIZE = 50` (em vez de 100), evite horários de pico (manhã EUA) e mantenha as palavras-chave razoavelmente concisas. Queries muito longas (muitos `OR`) tendem a falhar mais.

In [ ]:
# -----------------------------------------------------------------------------
# DEFINA AQUI O ESCOPO DA SUA COLEÇÃO - ADAPTADO PARA O SEU DOUTORADO (ZK-COMPLIANCE)
# -----------------------------------------------------------------------------

# Palavras-chave / frases do seu pré-projeto. Serão buscadas em título e abstract.
KEYWORDS = [
    "zero-knowledge",
    "zk-snark",
    "zk-stark",
    "data governance",
    "privacy-preserving auditing",
    "cryptographic compliance",
    "verifiable auditing"
]

# Categorias do ArXiv cruciais para Segurança/Criptografia e Banco de Dados.
CATEGORIES = ["cs.CR", "cs.DB"]

# Janela temporal baseada no amadurecimento das ZKPs em auditorias corporativas.
YEAR_FROM = 2018
YEAR_TO = 2026

# Tamanho-alvo aproximado da coleção conforme exigência do enunciado.
TARGET_SIZE = 1200

# Tamanho de cada página retornada pela API (mantido em 50 por robustez).
PAGE_SIZE = 50

# Caminho de saída da coleção bruta adaptado para salvar direto no seu Google Drive.
from pathlib import Path
OUTPUT_DIR = Path("/content/drive/MyDrive/Trabalho_IA/data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RAW_PATH = OUTPUT_DIR / "arxiv_raw.jsonl"
CORPUS_PATH = OUTPUT_DIR / "corpus.jsonl"

print("Configuração do ZK-Compliance carregada com sucesso.")
print(" - Keywords  :", KEYWORDS)
print(" - Categories:", CATEGORIES)
print(" - Years     :", YEAR_FROM, "->", YEAR_TO)
print(" - Target    :", TARGET_SIZE)
print(" - PageSize  :", PAGE_SIZE)

Configuração do ZK-Compliance carregada com sucesso.
 - Keywords  : ['zero-knowledge', 'zk-snark', 'zk-stark', 'data governance', 'privacy-preserving auditing', 'cryptographic compliance', 'verifiable auditing']
 - Categories: ['cs.CR', 'cs.DB']
 - Years     : 2018 -> 2026
 - Target    : 1200
 - PageSize  : 50


## 3. Coleta paginada via API do ArXiv (com retry/backoff)

A função abaixo monta uma *query* combinando palavras-chave (com `OR`) e categorias (com `OR`), e itera sobre os resultados em ordem decrescente de data de submissão.

**Como ela trata erros HTTP 429 (rate limit) e 503 (serviço indisponível):**
- Configura o cliente do `arxiv` com `delay_seconds=5` e `num_retries=8` --- mais paciente que o padrão.
- Encapsula a iteração em um *loop externo* com **backoff exponencial**: 60s → 120s → 240s → 480s → 600s entre tentativas.
- Mantém um *offset* de onde paramos: se cair, retoma a partir do mesmo ponto (sem re-baixar tudo).
- Salva incrementalmente em `arxiv_raw.jsonl` e *deduplica* por `arxiv_id`. **Você pode rodar a célula de coleta várias vezes** --- o progresso é preservado.

**Se mesmo assim não funcionar:** espere 15–30 minutos e tente de novo (o seu IP pode estar temporariamente bloqueado), ou reduza `PAGE_SIZE` para 25 e/ou diminua o número de palavras-chave.

In [ ]:
def build_query(keywords, categories):
    """Monta a string de query no formato esperado pela API do ArXiv."""
    kw_part = " OR ".join([f'all:"{k}"' for k in keywords]) if keywords else ""
    cat_part = " OR ".join([f"cat:{c}" for c in categories]) if categories else ""

    parts = [p for p in [f"({kw_part})" if kw_part else "",
                          f"({cat_part})" if cat_part else ""] if p]
    return " AND ".join(parts) if parts else "all:*"


QUERY = build_query(KEYWORDS, CATEGORIES)
print("Query final:\n", QUERY)

Query final:
 (all:"zero-knowledge" OR all:"zk-snark" OR all:"zk-stark" OR all:"data governance" OR all:"privacy-preserving auditing" OR all:"cryptographic compliance" OR all:"verifiable auditing") AND (cat:cs.CR OR cat:cs.DB)


In [ ]:
def already_collected_ids(path: Path) -> set:
    """Lê o arquivo .jsonl (se existir) e retorna os IDs já salvos."""
    if not path.exists():
        return set()
    ids = set()
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                ids.add(json.loads(line)["arxiv_id"])
            except Exception:
                continue
    return ids


def collect_arxiv(query, target_size, page_size, year_from, year_to,
                   out_path: Path,
                   max_outer_attempts: int = 6,
                   initial_backoff_seconds: int = 60):
    """Coleta artigos do ArXiv, salvando incrementalmente em JSONL.

    Robusto contra erros HTTP 429/503 da API do ArXiv:
    - Retry interno do cliente `arxiv` (delay_seconds=5, num_retries=8).
    - Loop externo com backoff exponencial caso a iteração falhe.
    - Retoma do mesmo offset após cada falha, sem re-baixar páginas inteiras.
    """
    client = arxiv.Client(page_size=page_size, delay_seconds=15, num_retries=15)

    seen = already_collected_ids(out_path)
    print(f"Já coletados anteriormente: {len(seen)} artigos.")

    saved_this_run = 0
    offset = 0          # quantos resultados já consumimos da API
    outer_attempt = 0   # quantas vezes o loop externo tentou

    while len(seen) < target_size and outer_attempt < max_outer_attempts:
        try:
            search = arxiv.Search(
                query=query,
                max_results=target_size * 3,  # 3x para sobrar margem após filtros
                sort_by=arxiv.SortCriterion.SubmittedDate,
                sort_order=arxiv.SortOrder.Descending,
            )

            print(f"\nIniciando/retomando do offset={offset} "
                  f"(salvos={len(seen)}, meta={target_size}).")

            with open(out_path, "a", encoding="utf-8") as f:
                pbar = tqdm(initial=len(seen), total=target_size,
                            desc="coletando")
                for result in client.results(search, offset=offset):
                    offset += 1
                    year = result.published.year if result.published else None
                    if year_from is not None and (year is None or year < year_from):
                        continue
                    if year_to is not None and (year is None or year > year_to):
                        continue

                    arxiv_id = result.get_short_id().split("v")[0]
                    if arxiv_id in seen:
                        continue

                    record = {
                        "arxiv_id": arxiv_id,
                        "title": (result.title or "").strip(),
                        "abstract": (result.summary or "").strip().replace("\n", " "),
                        "authors": [a.name for a in result.authors],
                        "categories": list(result.categories or []),
                        "primary_category": result.primary_category,
                        "published": result.published.isoformat() if result.published else None,
                        "updated": result.updated.isoformat() if result.updated else None,
                        "doi": result.doi,
                        "pdf_url": result.pdf_url,
                        "entry_id": result.entry_id,
                    }
                    f.write(json.dumps(record, ensure_ascii=False) + "\n")
                    f.flush()
                    seen.add(arxiv_id)
                    saved_this_run += 1
                    pbar.update(1)
                    pbar.set_postfix(offset=offset)

                    if len(seen) >= target_size:
                        break
                pbar.close()

            # Se chegou aqui sem exception, terminou normalmente.
            break

        except Exception as e:
            outer_attempt += 1
            wait = min(initial_backoff_seconds * (2 ** (outer_attempt - 1)), 600)
            print(f"\n[aviso] coleta interrompida (tentativa {outer_attempt}/"
                  f"{max_outer_attempts}): {type(e).__name__}: {e}")
            print(f"[aviso] aguardando {wait}s antes de retomar do offset={offset}...")
            for _ in tqdm(range(wait), desc="backoff", leave=False):
                time.sleep(1)

    print(f"\nColeta finalizada. Total acumulado em {out_path}: {len(seen)} artigos.")
    if len(seen) < target_size:
        print(f"[atenção] Não atingiu a meta de {target_size} (parou em {len(seen)}).")
        print( "[atenção] Você pode rodar esta célula novamente para continuar de onde parou.")
    return len(seen)

In [ ]:
import arxiv
import json
import time
from datetime import datetime
from pathlib import Path
import pandas as pd
from tqdm.notebook import tqdm  # Isso define o tqdm perfeitamente para o ambiente do Colab!

# Agora rodamos a função com todas as ferramentas carregadas na memória
n = collect_arxiv(
    query=QUERY,
    target_size=TARGET_SIZE,
    page_size=PAGE_SIZE,
    year_from=YEAR_FROM,
    year_to=YEAR_TO,
    out_path=RAW_PATH,
)
n

Já coletados anteriormente: 673 artigos.

Iniciando/retomando do offset=0 (salvos=673, meta=1200).


coletando:  56%|#####6    | 673/1200 [00:00<?, ?it/s]


Coleta finalizada. Total acumulado em /content/drive/MyDrive/Trabalho_IA/data/arxiv_raw.jsonl: 673 artigos.
[atenção] Não atingiu a meta de 1200 (parou em 673).
[atenção] Você pode rodar esta célula novamente para continuar de onde parou.


673

### O que fazer se a coleta continuar falhando

1. **Espere e tente de novo.** A API do ArXiv pode temporariamente bloquear o seu IP. Aguarde 15--30 min e rode a célula acima novamente --- ela vai retomar.
2. **Reduza a query.** Menos `OR` $\Rightarrow$ menos custo no servidor. Tente com 2--3 palavras-chave mais discriminativas.
3. **Reduza `PAGE_SIZE`** para 25 (mais requisições, cada uma mais leve).
4. **Aumente o `delay_seconds`** do `arxiv.Client` para 10 ou 15 segundos.
5. **Plano B: use o *bulk access* do ArXiv** (S3 / Kaggle dump) --- baixa em uma vez só, sem rate limit, ao custo de não ter os artigos mais recentes do dia.
6. **Plano C: troque a fonte** para Semantic Scholar, OpenAlex ou DBLP (todas listadas na Seção 5 do enunciado).

## 4. Normalização e deduplicação

A coleta bruta pode conter duplicatas (mesmo `arxiv_id` em diferentes versões) ou registros sem abstract. Geramos uma versão limpa em `corpus.jsonl`.

In [ ]:
def load_jsonl(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]


raw = load_jsonl(RAW_PATH)
print("Registros brutos:", len(raw))

# Deduplicar por arxiv_id (mantém o mais recente por 'updated').
df = pd.DataFrame(raw)
df["updated_dt"] = pd.to_datetime(df["updated"], errors="coerce")
df = df.sort_values("updated_dt").drop_duplicates("arxiv_id", keep="last")

# Remover registros sem título ou sem abstract.
df = df[df["title"].str.len() > 0]
df = df[df["abstract"].str.len() > 50]  # abstracts muito curtos costumam ser ruído

print("Após deduplicação e limpeza:", len(df))
df.head(3)

Registros brutos: 673
Após deduplicação e limpeza: 673


,arxiv_id,title,abstract,authors,categories,primary_category,published,updated,doi,pdf_url,entry_id,updated_dt
670,1801.08737,Lattice-Based Group Signatures: Achieving Full...,"In this work, we provide the first lattice-bas...","[San Ling, Khoa Nguyen, Huaxiong Wang, Yanhong...",[cs.CR],cs.CR,2018-01-26T10:09:03+00:00,2018-01-26T10:09:03+00:00,None,https://arxiv.org/pdf/1801.08737v1,http://arxiv.org/abs/1801.08737v1,2018-01-26 10:09:03+00:00
668,1802.05004,Zero-Knowledge Password Policy Check from Latt...,Passwords are ubiquitous and most commonly use...,"[Khoa Nguyen, Benjamin Hong Meng Tan, Huaxiong...",[cs.CR],cs.CR,2018-02-14T09:39:10+00:00,2018-02-14T09:39:10+00:00,None,https://arxiv.org/pdf/1802.05004v1,http://arxiv.org/abs/1802.05004v1,2018-02-14 09:39:10+00:00
667,1802.07023,BAN-GZKP: Optimal Zero Knowledge Proof based S...,BANZKP is the best to date Zero Knowledge Proo...,"[Gewu Bu, Maria Potop-Butucaru]","[cs.NI, cs.CR]",cs.NI,2018-02-20T09:19:11+00:00,2018-02-20T09:19:11+00:00,None,https://arxiv.org/pdf/1802.07023v1,http://arxiv.org/abs/1802.07023v1,2018-02-20 09:19:11+00:00


## 5. Salvamento em `corpus.jsonl`

Este é o arquivo que você vai usar nas próximas etapas (BM25, KNN, etc.).

In [ ]:
cols = ["arxiv_id", "title", "abstract", "authors", "categories",
         "primary_category", "published", "doi", "pdf_url"]

with open(CORPUS_PATH, "w", encoding="utf-8") as f:
    for _, row in df[cols].iterrows():
        f.write(json.dumps(row.to_dict(), ensure_ascii=False) + "\n")

print(f"Corpus salvo em: {CORPUS_PATH} ({len(df)} documentos).")

Corpus salvo em: /content/drive/MyDrive/Trabalho_IA/data/corpus.jsonl (673 documentos).


## 6. Verificação rápida da coleção (*sanity checks*)

Antes de partir para a recuperação, dê uma olhada na distribuição temporal, por categoria, e em alguns exemplos. Erros mais comuns: filtros muito apertados (corpus minúsculo), filtros muito frouxos (corpus contaminado com áreas alheias).

In [ ]:
df["year"] = pd.to_datetime(df["published"], errors="coerce").dt.year
print("Distribuição por ano:")
print(df["year"].value_counts().sort_index())
print("\nDistribuição por categoria primária:")
print(df["primary_category"].value_counts().head(10))

Distribuição por ano:
year
2018     23
2019     40
2020     43
2021     48
2022     57
2023     94
2024    107
2025    174
2026     87
Name: count, dtype: int64

Distribuição por categoria primária:
primary_category
cs.CR       502
quant-ph     44
cs.DB        29
cs.LG        25
cs.CY         9
cs.NI         8
cs.SE         8
cs.CC         7
cs.DC         7
cs.AR         7
Name: count, dtype: int64


In [ ]:
# Exemplos aleatórios para inspeção manual.
for _, row in df.sample(min(3, len(df)), random_state=42).iterrows():
    print("=" * 80)
    print("ID      :", row["arxiv_id"])
    print("Title   :", row["title"])
    print("Cats    :", row["categories"])
    print("Year    :", row["year"])
    print("Abstract:", row["abstract"][:400], "...")

ID      : 2305.16868
Title   : Location-aware Verification for Autonomous Truck Platooning Based on Blockchain and Zero-knowledge Proof
Cats    : ['cs.NI', 'cs.CR', 'cs.DC']
Year    : 2023
Abstract: Platooning technologies enable trucks to drive cooperatively and automatically, which bring benefits including less fuel consumption, more road capacity and safety. In order to establish trust during dynamic platoon formation, ensure vehicular data integrity, and guard platoons against potential attackers, it is pivotal to verify any given vehicle's identity information before granting it access t ...
ID      : 2405.04528
Title   : Implementing ISO/IEC TS 27560:2023 Consent Records and Receipts for GDPR and DGA
Cats    : ['cs.CR']
Year    : 2024
Abstract: The ISO/IEC TS 27560:2023 Privacy technologies - Consent record information structure provides guidance for the creation and maintenance of records regarding consent as machine-readable information. It also provides guidance on the use of 